# Reinforcement Learning

# 6. Bandit Algorithms

This notebook presents **multi-armed bandit** algorithms.

In [1]:
import numpy as np
from matplotlib import pyplot as plt

In [2]:
from model import Environment
from agent import Agent

## Multi-Armed Bandit

Multi-armed bandits are single-state models with random rewards.

In [3]:
class MAB(Environment):
    """Multi-Armed Bandit environnement.
    
    Parameters
    ----------
    distribution: string
        Reward distribution (Bernoulli, Uniform or Gaussian)
    params: list
        List of parameters (one per action)
        Example for Bernoulli (mean): [0.4, 0.5, 0.6]
        Example for Uniform (low, high): [(-1, 1), (0, 1), (-1, 2)]
        Example for Gaussian (mean, variance): [(0, 1), (1, 2), (-1, 1)]
    """

    def __init__(self, distribution='bernoulli', params=[0.4, 0.6]):        
        if type(distribution) != str:
            raise ValueError("The parameter 'distribution' must be a string: either 'bernoulli', 'uniform' or 'gaussian'.")
        self.distribution = distribution.lower()
        self.params = params
         
    @staticmethod
    def get_states():
        """Single state."""
        return [None]
    
    def get_actions(self):
        """One action per arm."""
        actions = [action for action, _ in enumerate(self.params)]
        return actions
    
    def get_reward(self, action):
        """Random reward. The parameter depends on the action."""
        if self.distribution == 'bernoulli':
            return np.random.random() < self.params[action]
        if self.distribution == 'uniform':
            low, high = self.params[action]
            return np.random.uniform(low, high)
        if self.distribution in ['gaussian', 'normal']:
            mean, std = self.params[action]
            return np.random.normal(mean, std)
        raise ValueError('Unknown distribution.')
        
    def get_model(self):
        raise ValueError('Not available.')
        
    def step(self, action):
        stop = False
        reward = self.get_reward(action)
        return reward, stop

In [4]:
class Bandit(Agent):
    """Bandit algorithm with random policy. 

    Parameters
    ----------
    model : object of class MAB
        The model.
    init_value : float
        Initial value of the action-value function.
    init_count : int
        Initial count of the action-count function.
    """
    
    def __init__(self, model, init_value=0, init_count=0):
        if not isinstance(model, MAB):
            raise ValueError('The model must be a multi-armed bandit.')
        self.model = model
        self.policy = self.random_policy
        actions = model.get_actions()
        self.values = len(actions) * [init_value]
        self.counts = len(actions) * [init_count]

    def get_actions(self, state=None):
        """Get all possible actions."""
        return self.model.get_actions()
    
    def get_episode(self, horizon=100):
        """Get the rewards for an episode and update the values."""
        state = None
        rewards = []
        for t in range(horizon):
            action = self.get_action(state)
            reward, _ = self.model.step(action)
            rewards.append(reward)
            self.counts[action] += 1
            diff = reward - self.values[action]
            # update by temporal difference
            self.values[action] += diff / self.counts[action]
        return rewards    

In [ ]:
model = MAB()
print("Distribution:", model.distribution)
print("Parameters:", model.params)

In [ ]:
agent = Bandit(model)
print("Actions:", agent.get_actions())

In [ ]:
rewards = agent.get_episode(horizon=100)
print("Gain:", np.mean(rewards))
print("Values:", agent.values)
print("Counts:", agent.counts)

## The $\varepsilon$-greedy policy

In [8]:
class Greedy(Bandit):
    """Bandit algorithm with epsilon-greedy policy. 

    Parameters
    ----------
    model : object of class MAB
        The model.
    epsilon : float in [0, 1]
        Exploration rate.
    init_value : float
        Initial value of the action-value function.
    init_count : int
        Initial count of the action-count function.
    """
    
    def __init__(self, model, epsilon=0.1, init_value=0, init_count=0):
        super(Greedy, self).__init__(model, init_value, init_count) 
        self.epsilon = epsilon

    def get_action(self, state=None):
        """Get action with eps-greedy policy."""
        actions = self.get_actions()
        if np.random.random() > self.epsilon:
            # select the best action(s) with probability 1 - epsilon
            values = np.array(self.values)
            actions = np.flatnonzero(values==np.max(values))
        return np.random.choice(actions)


In [ ]:
agent = Greedy(model)
rewards = agent.get_episode(horizon=100)
print("Gain:", np.mean(rewards))
print("Values:", agent.values)
print("Counts:", agent.counts)

## To do 

* Compute the expected **gain** for $\varepsilon = 0.2$ and check your result by simulation.<br>**Hint:** You might adapt the time horizon.
* Observe the phenomenon of **optimism in face of incertainty** when $\varepsilon = 0$.<br>**Hint:** You might adapt the parameters ``init_value`` and ``init_count``.

### Expected gain for $\epsilon=0.2$

$$
E[G] = \left(\frac{\epsilon}{2} + (1-\epsilon)\right) \cdot Q(B) + \frac{\epsilon}{2} \cdot Q(A)\\
E[G] = \left(\frac{1}{10} + \frac{4}{5}\right) \cdot Q(B) + \frac{1}{10} \cdot Q(A)\\
E[G] = \left(\frac{1}{10} + \frac{4}{5}\right) \cdot \frac{3}{5} + \frac{1}{10} \cdot \frac{2}{5}\\
E[G] = \frac{29}{50}\\
E[G] = 0.58
$$

### Simulation for $\epsilon=0.2$

In [ ]:
agent = Greedy(model, epsilon=0.2)
rewards = agent.get_episode(horizon=10000)
print("Gain:", np.mean(rewards))
print("Values:", agent.values)
print("Counts:", agent.counts)

### Expected gain for $\epsilon=0$

$$
E[G] = \left(\frac{\epsilon}{2} + (1-\epsilon)\right) \cdot Q(B) + \frac{\epsilon}{2} \cdot Q(A)\\
E[G] = Q(B)\\
E[G] = 0.60
$$

### Simulation for $\epsilon=0$

In [ ]:
agent = Greedy(model, epsilon=0, init_value=1, init_count=1) 
rewards = agent.get_episode(horizon=10000)
print("Gain:", np.mean(rewards))
print("Values:", agent.values)
print("Counts:", agent.counts)

## The UCB policy

We now consider the UCB (Upper Confidence Bound) policy.

## To do

* Complete and test the agent ``UCB`` below.
* Plot the **regret** with respect to the time horizon, and compare with the $\varepsilon$-greedy policy for different values of $\varepsilon$.
* Repeat this experiment for the other models (uniform and Gaussian).<br> Interpret the results.
* Test the impact of the number of actions.

In [12]:
class UCB(Bandit):
    """Bandit algorithm with UCB policy. 

    Parameters
    ----------
    model : object of class MAB
        The model.
    const : float in [0, 1]
        Multiplicative constant for the UCB bonus.
    init_value : float
        Initial value of the action-value function.
    init_count : int
        Initial count of the action-count function.
    """
    
    def __init__(self, model, const=1, init_value=0, init_count=0):
        super(UCB, self).__init__(model, init_value, init_count) 
        self.const = const

    def get_action(self, state=None):
        """Get action with UCB policy."""
        values = np.array(self.values)
        counts = np.array(self.counts)
        actions = self.get_actions()
        # to be modified
        # not visited actions
        not_visited = np.flatnonzero(counts==0)
        if len(not_visited):
            return np.random.choice(not_visited)
        # visited actions
        ucb = values + self.const * np.sqrt(np.log(np.sum(counts)) / counts)
        actions = np.flatnonzero(ucb==np.max(ucb))
        return np.random.choice(actions)

In [ ]:
agent = UCB(model)
rewards = agent.get_episode(horizon=10000)
regret = np.cumsum(np.max(model.params) - rewards)
print("Gain:", np.mean(rewards))
print("Values:", agent.values)
print("Counts:", agent.counts)

## Regret analysis

In [14]:
def calculate_regret(model, rewards):
    if model.distribution == 'uniform':
        means = np.array([(a + b) / 2 for a, b in model.params])
    elif model.distribution in ['gaussian', 'normal']:
        means = np.array([mu for mu, _ in model.params])
    elif model.distribution == 'bernoulli':
        means = np.array(model.params)
    else:
        raise ValueError("Unknown distribution")
    
    regret = np.cumsum(np.max(means) - rewards)
    return regret

In [15]:
def run_experiment_eg(distribution, params, epsilons, runs=5, horizon=10000):
    model = MAB(distribution=distribution, params=params)
    regrets_all = {}

    for epsilon in epsilons:
        regrets = []
        for _ in range(runs):
            if epsilon == 0:
                agent = Greedy(model, epsilon=epsilon, init_value=1, init_count=1)
            else:
                agent = Greedy(model, epsilon=epsilon)
            rewards = agent.get_episode(horizon=horizon)
            regret = calculate_regret(model, rewards)
            regrets.append(regret)
        regrets_all[epsilon] = np.mean(regrets, axis=0)

    return regrets_all

def run_experiment_ucb(distribution, params, runs=5, horizon=10000):
    model = MAB(distribution=distribution, params=params)
    
    regrets = []
    for _ in range(runs):
        agent = UCB(model)
        rewards = agent.get_episode(horizon=horizon)
        regret = calculate_regret(model, rewards)
        regrets.append(regret)
    regret_ucb = np.mean(regrets, axis=0)

    return regret_ucb

In [16]:
def plot_regret(ax, regrets, title):
    if 'ucb' in regrets:
        ax.plot(regrets['ucb'], label='UCB')
    if 'eg' in regrets:
        for epsilon, regret in regrets['eg'].items():
            ax.plot(regret, label='epsilon = ' + str(epsilon))
    if 'ts' in regrets:
        ax.plot(regrets['ts'], label='TS')
    ax.legend()
    ax.set_xlabel('Time steps')
    ax.set_ylabel('Regret')
    ax.set_title(title)
    return ax

### Epsilon value impact

In [17]:
epsilons = [0, 0.1, 0.2]
dist_params = {'bernoulli': [0.4, 0.6], 'uniform': [(-1, 1), (0, 1), (-1, 2)], 'normal': [(0, 1), (1, 2), (-1, 1)]}
runs = 25
horizon = 2500

dist_regrets = {}
for distribution in dist_params:
    # Epsilon-Greedy
    regrets_eg = run_experiment_eg(distribution, dist_params[distribution], epsilons, runs, horizon)

    # UCB
    regret_ucb = run_experiment_ucb(distribution, dist_params[distribution], runs, horizon)

    dist_regrets[distribution] = {'ucb': regret_ucb, 'eg': regrets_eg}

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

plot_regret(axs[0], dist_regrets['bernoulli'], 'Bernoulli')
plot_regret(axs[1], dist_regrets['uniform'], 'Uniform')
plot_regret(axs[2], dist_regrets['normal'], 'Normal')

plt.tight_layout()
plt.show()

**Interpretation on epsilon value impact**

Epsilon value of 0 has the most variance in regret. This is because it can go very well (pick the highest reward action) or very wrong (and pick the lowest reward action and stay there). When running the experiments various times one can observe that the line for epsilon = 0 is very close to 0 or be the highest line of all. UCB constantly outperforms the epsilon-greedy algoritm across distributions. For the epsilon comparison, a low epsilon (low exploration rate) yield the lowest regret constantly. However if the epsilon is very close to zero, the chances of having a very high regret increases due to a null exploration rate (UCB fixes this).

### Number of actions impact

#### Bernoulli

In [19]:
epsilons = [0.1, 0.2]
distribution = 'bernoulli'
action_params_ber = {
    2: [0.4, 0.6],
    5: [0.1, 0.2, 0.3, 0.4, 0.5],
    10: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
}
runs = 10
horizon = 4000

params_regret_ber = {}
for num_actions in action_params_ber:
    # Epsilon-Greedy
    regrets_eg = run_experiment_eg(distribution, action_params_ber[num_actions], epsilons, runs, horizon)

    # UCB
    regret_ucb = run_experiment_ucb(distribution, action_params_ber[num_actions], runs, horizon)
    
    params_regret_ber[num_actions] = {'ucb': regret_ucb, 'eg': regrets_eg}

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

plot_regret(axs[0], params_regret_ber[2], '2 actions')
plot_regret(axs[1], params_regret_ber[5], '5 actions')
plot_regret(axs[2], params_regret_ber[10], '10 actions')

fig.suptitle('Bernoulli Distribution')
plt.tight_layout()
plt.show()

#### Uniform

In [21]:
epsilons = [0.1, 0.2]
distribution = 'uniform'
action_params_unif = {
    2: [(-1, 1), (0, 1)],
    5: [(-1, 1), (0, 1), (-1, 2), (0, 3), (1, 2)],
    10: [(-1, 1), (0, 1), (-1, 2), (0, 3), (1, 2), (-2, 1), (0, 2), (1, 3), (-1, 3), (2, 3)]
}
runs = 10
horizon = 4000

params_regret_unif = {}
for num_actions in action_params_unif:
    # Epsilon-Greedy
    regrets_eg = run_experiment_eg(distribution, action_params_unif[num_actions], epsilons, runs, horizon)

    # UCB
    regret_ucb = run_experiment_ucb(distribution, action_params_unif[num_actions], runs, horizon)
    
    params_regret_unif[num_actions] = {'ucb': regret_ucb, 'eg': regrets_eg}

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

plot_regret(axs[0], params_regret_unif[2], '2 actions')
plot_regret(axs[1], params_regret_unif[5], '5 actions')
plot_regret(axs[2], params_regret_unif[10], '10 actions')

fig.suptitle('Uniform Distribution')
plt.tight_layout()
plt.show()

#### Normal

In [23]:
epsilons = [0.1, 0.2]
distribution = 'normal'
action_params_norm = {
    2: [(0, 1), (1, 2)],
    5: [(0, 1), (1, 2), (-1, 1), (2, 3), (3, 2)],
    10: [(0, 1), (1, 2), (-1, 1), (0, 2), (2, 3), (-2, 1), (0, 3), (3, 2), (-1, 4), (1, 1)]
}
runs = 10
horizon = 4000

params_regret_norm = {}
for num_actions in action_params_norm:
    # Epsilon-Greedy
    regrets_eg = run_experiment_eg(distribution, action_params_norm[num_actions], epsilons, runs, horizon)

    # UCB
    regret_ucb = run_experiment_ucb(distribution, action_params_norm[num_actions], runs, horizon)
    
    params_regret_norm[num_actions] = {'ucb': regret_ucb, 'eg': regrets_eg}

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

plot_regret(axs[0], params_regret_norm[2], '2 actions')
plot_regret(axs[1], params_regret_norm[5], '5 actions')
plot_regret(axs[2], params_regret_norm[10], '10 actions')

fig.suptitle('Normal Distribution')
plt.tight_layout()
plt.show()

**Interpretation on number of actions impact**

UCB algorithm constantly outperforms the epsilon-greedy algorithm in terms of regret across all number of actions and distributions (with some exceptions of eps=0). Epsilon = 0.1 generally performs better than epsilon = 0.2, becoming more evident with a higher number of actions (this is expected since the probability of exploration is higher and the probability of choosing the optimal action when exploring is lower). The number of actions increase the slope of the regret over time, this means that the model struggles more when more actions are possible and probably does not constantly choose the action with the highest expected regret or it takes it more time to do so. This is expected since a larger action space makes the problem harder, requiring more exploration to identify the optimal action.

The regret for epsilon-greedy grows approximately linearly with time steps especially with larger actions. On the other hand the UCB curve flattens or shows a curvature (see bernoulli distribution plots) which indicates sublinear regret i.e. the model learns the optimal action and sticks to it.

## Thompson sampling

Finally, we consider Thompson Sampling, where the mean rewards are considered as random and sampled according to the posterior distribution (Bayesian algorithm).

## To do

* Complete and test the agent ``TS`` below.
* Plot the **regret** with respect to the time horizon, and compare with the UCB policy for different models.
* Which algorithm is the more efficient?<br> Comment your results.

In [25]:
class TS(Bandit):
    """Bandit algorithm with Thompson sampling. 

    Parameters
    ----------
    model : object of class MAB
        The model.
    """
    
    def __init__(self, model):
        super(TS, self).__init__(model) 
        self.distribution = model.distribution
            
    def get_action(self, state=None):
        """Get action with TS policy."""
        values = np.array(self.values)
        counts = np.array(self.counts)
        if self.distribution == 'bernoulli':
            # to be modified
            alpha = values * counts + 1
            beta = counts - values * counts + 1
            samples = np.random.beta(alpha, beta)
        else:
            # to be modified
            mean = values * counts / (counts + 1)
            std = 1 / np.sqrt(counts + 1)
            samples = np.random.normal(mean, std)
        return np.argmax(samples)


In [26]:
def run_experiment_ts(distribution, params, runs=5, horizon=10000):
    model = MAB(distribution=distribution, params=params)
    
    regrets = []
    for _ in range(runs):
        agent = TS(model)
        rewards = agent.get_episode(horizon=horizon)
        regret = calculate_regret(model, rewards)
        regrets.append(regret)
    regret_ts = np.mean(regrets, axis=0)

    return regret_ts

### UCB vs TS comparison

#### Bernoulli

In [27]:
distribution = 'bernoulli'
action_params_ber = {
    2: [0.4, 0.6],
    5: [0.1, 0.2, 0.3, 0.4, 0.5],
    10: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
}
runs = 25
horizon = 4000

ucbts_regret_ber = {}
for num_actions in action_params_ber:
    # UCB
    regret_ucb = run_experiment_ucb(distribution, action_params_ber[num_actions], runs, horizon)

    # TS
    regrets_ts = run_experiment_ts(distribution, action_params_ber[num_actions], runs, horizon)
    
    ucbts_regret_ber[num_actions] = {'ucb': regret_ucb, 'ts': regrets_ts}

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

plot_regret(axs[0], ucbts_regret_ber[2], '2 actions')
plot_regret(axs[1], ucbts_regret_ber[5], '5 actions')
plot_regret(axs[2], ucbts_regret_ber[10], '10 actions')

fig.suptitle('Bernoulli Distribution')
plt.tight_layout()
plt.show()

#### Uniform

In [29]:
distribution = 'uniform'
action_params_unif = {
    2: [(-1, 1), (0, 1)],
    5: [(-1, 1), (0, 1), (-1, 2), (0, 3), (1, 2)],
    10: [(-1, 1), (0, 1), (-1, 2), (0, 3), (1, 2), (-2, 1), (0, 2), (1, 3), (-1, 3), (2, 3)]
}
runs = 25
horizon = 4000

ucbts_regret_unif = {}
for num_actions in action_params_unif:
    # UCB
    regret_ucb = run_experiment_ucb(distribution, action_params_unif[num_actions], runs, horizon)

    # TS
    regrets_ts = run_experiment_ts(distribution, action_params_unif[num_actions], runs, horizon)
    
    ucbts_regret_unif[num_actions] = {'ucb': regret_ucb, 'ts': regrets_ts}

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

plot_regret(axs[0], ucbts_regret_unif[2], '2 actions')
plot_regret(axs[1], ucbts_regret_unif[5], '5 actions')
plot_regret(axs[2], ucbts_regret_unif[10], '10 actions')

fig.suptitle('Uniform Distribution')
plt.tight_layout()
plt.show()

#### Normal

In [35]:
distribution = 'normal'
action_params_norm = {
    2: [(0, 1), (1, 2)],
    5: [(0, 1), (1, 2), (-1, 1), (2, 2), (-1, 1)],
    10: [(0, 1), (1, 2), (-1, 1), (0, 2), (2, 3), (-2, 1), (0, 3), (3, 2), (-1, 4), (1, 1)]
}
runs = 25
horizon = 4000

ucbts_regret_norm = {}
for num_actions in action_params_norm:
    # UCB
    regret_ucb = run_experiment_ucb(distribution, action_params_norm[num_actions], runs, horizon)

    # TS
    regrets_ts = run_experiment_ts(distribution, action_params_norm[num_actions], runs, horizon)
    
    ucbts_regret_norm[num_actions] = {'ucb': regret_ucb, 'ts': regrets_ts}

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

plot_regret(axs[0], ucbts_regret_norm[2], '2 actions')
plot_regret(axs[1], ucbts_regret_norm[5], '5 actions')
plot_regret(axs[2], ucbts_regret_norm[10], '10 actions')

fig.suptitle('Normal Distribution')
plt.tight_layout()
plt.show()

**Interpretation on UCB and TS comparison**

TS is the more efficient algorithm. It outperforms UCB in every distribution and number of actions. In most cases, when the number of actions increases the gap between UCB and TS also increases with the time steps. Here we can also observe a subliner regret curve for both algorithms. 